# Day 2 错题本｜两轮题目 + 答案

> **第一轮**：57 / 100（概念 19 ｜ 计算 21 ｜ 编程 4 ｜ 找 bug 13）
> **第二轮**：**76 / 100**（概念 24 ｜ 计算 23 ｜ 编程 15 ｜ 找 bug 14）
> **读法**：<span style="color:#c62828">红色 = 答错</span> ｜ <span style="color:#e65100">橙色 = 半对</span> ｜ 黑色 = 正确

**文件结构**

1. 第一轮｜首考卷（每题：题目 → 你写的 → 正确答案）
2. 第二轮｜重考卷（每题：题目 → 你写的 → 正确答案）
3. 仍未解决的 4 个洞（含 20 分钟补漏方案）
4. 重做清单

> 合并自 `day2ans.ipynb` + `day2_retest.ipynb`。

---


## 第一轮 A 组｜概念（19 / 30）

### A1 矩阵的列与交换律 — <span style="color:#e65100">【半对】</span>

**题目**：矩阵的每一列在几何上代表什么？由此解释为什么矩阵乘法不满足交换律。

**你写的**：变换后基向量的位置；AB 是先做 B 变换再做 A 变换，故矩阵乘法不满足交换律。

**答案**：前半句对——矩阵的每一列就是变换后基向量的落点（列即 î、ĵ 的新位置）。
缺的一环是：AB 和 BA 是**两次顺序不同的变换**，而变换的先后一般不能互换——"先旋转 90° 再横向拉伸"≠"先横向拉伸再旋转 90°"，所以 AB ≠ BA。

---

### A2 找函数三步曲 — ✓

**题目**：复述三步曲，并说明每步是"人"还是"机器"的工作。

**答案**：① 定义模型——人设计函数形式（如 y = wx + b），机器负责找出 w、b；② 定义损失——人；③ 优化参数——机器（梯度下降）。你答得完整。

---

### A3 Model Bias — <span style="color:#e65100">【半对】</span>

**题目**：线性模型的 Model Bias 是什么？深度学习用哪两件事打破它？

**你写的**：无法用线性的直线来模拟曲线；用简单的曲线来分段拟合，用很多层来拟合。

**答案**：方向对，补齐关键词：
Model Bias = 模型能表达的函数集合里根本没有"真函数"，症状是 **training loss 怎么调都下不去**。
破局两件事：① **非线性激活函数**（ReLU / Sigmoid）——两个 ReLU 相减就是一个"折角"，多个叠加成分段折线；② **多层堆叠**——深层复用浅层的简单特征，组合出复杂曲线（比单纯做宽更省参数）。

---

### A4 逐元素 vs 矩阵乘 — <span style="color:#c62828">【错】</span>

**题目**：列出至少 4 个逐元素运算，并说明 `*` 和 `@` 的区别。

**你写的**："+ - * /；`*` 是叉乘，是逐元素相乘，`@` 是点乘"——一句话里出现两个互相矛盾的词，说明这是印象不是概念。

**答案**：

| 写法 | 是什么 | 要求 |
| --- | --- | --- |
| `x + y`、`x - y`、`x * y`、`x / y`、`x ** y`、`torch.exp(x)` | 逐元素运算 | 形状相同，或可广播 |
| `x @ y` / `torch.matmul(x, y)` | **矩阵乘法** | 必须满足 (m,k) @ (k,n) → (m,n) |
| `torch.dot(x, y)` / `(x * y).sum()` | **点积**（结果是标量） | 一维向量 |
| `torch.cross(x, y)` | 叉乘（结果是向量） | 三维向量专用 |

一句话记牢：`*` 是"位置对着位置"，`@` 是"行乘列求和"。`*` 常常能跑通但算错，`@` 维度不对会直接报错——所以 shape bug 多半出在 `*` 上。

---

### A5 cat 与原地修改 — <span style="color:#e65100">【半对】</span>

**题目**：`cat` 的 dim=0 / dim=1 分别得到什么形状？为什么 `Y = Y + X` 不改变原对象，而 `Z[:] = X + Y` 会？

**你写的**：原理说对了（新建副本 vs 原地修改），但没给出形状。

**答案**：X 是 (3,4)、Y 是 (3,4) 时：

- `torch.cat((X, Y), dim=0)` → **(6,4)**：竖直叠放，样本数增加。
- `torch.cat((X, Y), dim=1)` → **(3,8)**：水平拼接，特征数增加。
- `Y = Y + X`：新建一个张量，把名字 `Y` 重新指向新对象，`id(Y)` 改变，原对象没动。
- `Z[:] = X + Y`：把结果**拷贝进 Z 已有的内存**，`id(Z)` 不变，所以是原地修改。

---

### A6 backward 与标量 — <span style="color:#e65100">【半对】</span>

**题目**：为什么 `.backward()` 要求输出是标量？非标量时有哪两种写法、各自用途？

**你写的**："只有标量才能求导"；两种写法 `y.sum().backward()` / `y.backward(torch.ones_like(x))`。

**答案**：两种写法记对了，但原因说反了——**不是"只有标量能求导"**，而是链式法则需要一个起点，标量时这个起点天然是 ∂L/∂L = 1。

| 写法 | 得到什么 | 用途 |
| --- | --- | --- |
| `y.sum().backward()` / `.mean()` | 各分量导数之和 | 训练里的默认做法 |
| `y.backward(torch.ones_like(x))` | 沿"全 1 方向"的方向导数（VJP）；把 `ones` 换成 one-hot，可逐个取出 Jacobian 的每一列 | 求 Jacobian、高阶导、HVP |

---

### A7 detach 与 requires_grad — <span style="color:#c62828">【错】</span>

**题目**：`.detach()` 切断了什么？举一个必须用它的场景；它和 `requires_grad=False` 的区别？

**你写的**：对该元素算梯度；把某个中间结果当常数用。

**答案**：

- `detach()` 返回一个**与计算图断开**的新张量：和原张量共享底层数据，但不带 `grad_fn`，反向传播走到这里就终止（"剪断求导电线"）。
- 必须用它的场景：① 把带梯度的张量转成 numpy 或拿去画图（`features.detach().numpy()`）；② 把某个中间结果当常数用（例如目标网络的 target）。
- 与 `requires_grad=False` 的区别：后者是"**这个参数不参与训练**"，前者是"**这条计算路径到此为止**"，语义不同。

---

### A8 动态计算图 — <span style="color:#e65100">【半对】</span>

**题目**：动态图"边跑边画"是什么意思？为什么 Python 原生 if / while 也能正确求导？

**你写的**：前向执行到什么地方，就把该路径记录下来；因为真实轨迹被记录下来了。

**答案**：方向对，可以更狠：前向执行到哪个算子，就把哪个算子记进计算图，图的形状由**运行时**决定。`while` 每迭代一次就产生一个新的乘法节点、被记一次，循环几次就记几个节点，所以反向传播只要沿这条真实记录的链回传即可。（对比静态图：必须先把整张图固定死，再往里灌数据。）

---

### A9 梯度累加 — <span style="color:#e65100">【半对】</span>

**题目**：梯度为什么默认累加？忘记清零时 `optimizer.step()` 用的是什么？

**你写的**：第一问"不知道"；第二问答对了（这批加上批）。

**答案**：

- 为什么累加是设计选择：多个 loss / 多个分支的梯度可以直接相加（多任务训练、RNN 跨时间步），也支撑"梯度累积"技巧。
- 忘记清零 → `.grad` 是**历史所有批次梯度之和**，`step()` 用的就是它，等效学习率被放大、方向被旧数据污染，表现为 loss 抖动到发散（严格说不是"爆炸"，是累加成 k 倍）。

---

### A10 广播 — <span style="color:#e65100">【半对】</span>

**题目**：广播的对齐规则？为什么它容易造成静默 bug？

**你写的**：从右向左对齐；两个维度错误的向量进行计算可能会导致广播，从而得出错误答案但不报错。

**答案**：规则是——从**最右边**开始逐维对齐，每一维要么相等、要么其中一个是 1（缺失的维度视为 1），满足就能算。
静默 bug 的来源：形状**恰好符合广播规则**时不报错，却算出你没意识到的张量。例：`(16,1) - (16,)` → **(16,16)**。


## 第一轮 B 组｜计算（21 / 30）

### B1 形状三连 — <span style="color:#c62828">【错】</span>

**题目**：`x = torch.arange(12)`；求 `x.shape`、`x.numel()`、`reshape(3,4)` 后 `X` 的形状、`X[-1]`、`X[1:3]` 的形状。

**你写的**：(1, 12); 12; (3, 4); (1, 4); (2, 4)

**答案**：

| 表达式 | 形状 | 说明 |
| --- | --- | --- |
| `torch.arange(12).shape` | **(12,)** | 一维，不是 (1,12) |
| `numel()` | 12 | 元素个数 |
| `x.reshape(3, 4).shape` | (3, 4) | 只改视图，不改原张量 |
| `X[-1]` | **(4,)** | 单个整数索引 → 把那一维消掉 |
| `X[-1:]` | (1, 4) | 切片 → 保留维度 |
| `X[1:3]` | (2, 4) | 切片 |

**记忆点**：整数索引降维，切片不降维。这两者混起来，就是 B7 / B8 / B9 把"梯度"写成"梯度的和"的同一个病根。

---

### B2 索引赋值 — ✓

**题目**：`X[1,2] = 9` 后第 1 行内容；再 `X[0:2,:] = 12` 后 `X[1,3]` 的值。

**答案**：`[4, 5, 9, 7]`；`12`。全对。

---

### B3 广播结果 — ✓

**题目**：`a = arange(3).reshape(3,1)`、`b = arange(2).reshape(1,2)`，写出 `a + b`。

**答案**：

```
[[0, 1],
 [1, 2],
 [2, 3]]
```

补一句理由：3×1 与 1×2 每一维都满足"其中一个为 1 可拉伸"，结果取每一维的最大值 → (3,2)。

---

### B4 维度消消乐 — <span style="color:#e65100">【半对】</span>

**题目**：X 是 (16,3)、w 是 (3,1)、b 是标量 → `(X@w+b).shape`？w 写成 (3,) 呢？再与 (16,1) 的标签相减呢？

**你写的**：(16, 1)；(16,)；第三问"不懂"。

**答案**：

- `(X @ w + b).shape` → **(16, 1)**
- `w` 写成 `(3,)` 时 → **(16,)**，一维
- 与 `(16, 1)` 的标签相减 → **(16, 16)**：16 行的预测各减 16 个标签，**不报错**，Loss 变成 256 个元素的错矩阵，梯度全错

所以硬规矩是：标签和预测都显式写成 `(n, 1)`（`.reshape(-1, 1)`），别让一维张量混进 loss。这正是 day3 笔记里"矩阵维度消消乐"的坑。

---

### B5 逐元素运算 — <span style="color:#e65100">【半对】</span>

**题目**：`x = [1,2,4,8]`、`y = [2,2,2,2]`，写出五种运算和 `torch.exp(x)`。

**你写的**：只有 `x**y` 的第一个元素错（写成 2）。

**答案**：`x+y = [3,4,6,10]`；`x-y = [-1,0,2,6]`；`x*y = [2,4,8,16]`；`x/y = [0.5,1,2,4]`；`x**y = [1,4,16,64]`（1² = 1）；`torch.exp(x) = [e¹, e², e⁴, e⁸]`。

---

### B6 标量求导 — ✓

**题目**：`y = 2 * torch.dot(x, x)`，x = [0,1,2,3]，求 ∂y/∂x。

**答案**：y = 2Σxᵢ² ⇒ ∂y/∂xᵢ = 4xᵢ = **[0, 4, 8, 12]**，形状 (4,)。

---

### B7 梯度累加 — <span style="color:#c62828">【错】</span>

**题目**：`y = (x*x).sum()` 的梯度？不清零再跑一次？累加 k 次的通式？

**你写的**：12；24；12*(k-1)

**答案**：

- 梯度 = **2x = [0, 2, 4, 6]**（形状 (4,)，和 x 一致）
- 不清零再跑一次 → **[0, 4, 8, 12]**（两倍）
- 累加 k 次 → **2k·x**（逐元素）；其元素之和是 12k

<span style="color:#c62828">你把"梯度的和"当成了梯度。</span>**梯度永远是向量，形状与参数相同**；12 是元素之和，不是导数。你写的 `12(k-1)` 连和都不对（应是 12k）。

---

### B8 detach 后的梯度 — <span style="color:#e65100">【半对】</span>

**题目**：`y = x*x`、`u = y.detach()`、`z = (u*x).sum()`，求 ∂z/∂x，并解释为什么不是 3x²。

**你写的**：14；因为 detach 把 y 当常数用了。

**答案**：理由对，但答案形式错。u = detach(x²) = [0,1,4,9] 是常数，z = Σuᵢxᵢ，所以

**∂z/∂x = u = [0, 1, 4, 9] = x²**

不是 3x² 的原因：y 这一支被切断，只剩 x 这一支贡献 u；**不 detach** 才是 ∂(x²·x)/∂x = x² + 2x² = 3x² = [0, 3, 12, 27]。
（"14" 是你又把梯度求和了。）

---

### B9 非标量 backward — <span style="color:#e65100">【半对】</span>

**题目**：`y = x*x` 直接 `y.backward()` 会怎样？改成 `y.backward(torch.ones_like(x))` 呢？和 `y.sum().backward()` 的关系？

**你写的**：报错，矢量不能进行梯度计算; 12; 等价

**答案**：

- 直接 `y.backward()` → `RuntimeError: grad can be implicitly created only for scalar outputs`（措辞应改成"backward 需要标量起点"，而不是"矢量不能求导"）
- `y.backward(torch.ones_like(x))` → **x.grad = 2x = [0, 2, 4, 6]**
- 与 `y.sum().backward()` **完全等价**，这句你答对了

---

### B10 标量转换与 numpy — <span style="color:#e65100">【半对】</span>

**题目**：`a = torch.tensor([3.5])` 的 `item()`、`float()`、`int()`；`X.numpy()` 什么时候报错？

**你写的**：3.5, 3.5, 3；最后一问"不懂"。

**答案**：`a.item()` = 3.5（Python float）、`float(a)` = 3.5、`int(a)` = 3（截断）。
最后一问：`requires_grad=True` 的张量**不能**直接 `.numpy()`，会报 `Can't call numpy() on Tensor that requires grad`，要先 `.detach()`；如果在 GPU 上还要先 `.cpu()`，完整写法 `x.detach().cpu().numpy()`。


## 第一轮 C 组｜编程（4 / 20）

### C1 创建与 reshape — <span style="color:#e65100">【半对】</span>

**你写的**：能跑，但没发现输出里的问题。`x.reshape(3, 4)` 只是**返回**一个新张量，没被任何变量接住，所以第二个 `print(x.shape)` 打出来还是 `(12,)`——输出里两行完全一样就是证据。另外新建的 y、z 也没打印形状。

**要点**：`reshape` 不改原张量；要拿到结果必须赋值（`x2 = x.reshape(3, 4)`）。下面第一个 code cell 是参考实现。

### C2 按列标准化 — <span style="color:#c62828">【错】</span>

**你写的**：只有 `x[:]`，标准化没写。

**要点**：统计维是 `dim=0`（每个特征一列），结果是 (3,)，再靠广播铺回 (100,3)。`unbiased=False` 用 n 作除数（和 BatchNorm 一致），`True` 用 n−1。

### C3 带 while / if 的 f(a) — <span style="color:#c62828">【错】</span>

**你写的**：函数体是空的。

**要点**：答案模板就在你自己 day2 笔记的最后一个 code cell 里。关键是 `backward()` 之后 `a.grad == d / a` 必须成立。

### C4 gradcheck — <span style="color:#c62828">【错】</span>

**你写的**：空的。

**要点**：① 必须用 `dtype=torch.double`；② 它用有限差分对比解析梯度；③ 随机点正好落在 while / if 的**分支边界**附近时会偶发失败——微小扰动会改变循环次数或分支，数值梯度和解析梯度必然不一致。这是分段函数的天性，不是代码写错。

### C5 最小训练循环 — <span style="color:#e65100">【半对】</span>

**你写的**：只有前向 + 反向，三次 loss 都是 `22.14858627319336`。

**要点**：少的正是**清零**和**更新**这两步。loss 三次完全一样，就是"没有 step 就没有学习"的直接证据。补完后 loss 必须单调下降，并且 w、b 要收敛到真值 3 和 2。


In [ ]:
# C1 参考实现：reshape 不改原张量
import torch

x = torch.arange(12)
print("x        :", x.shape, x.numel())

x2 = x.reshape(3, 4)          # 必须接住返回值
print("x 还是   :", x.shape)    # torch.Size([12])
print("x2       :", x2.shape)   # torch.Size([3, 4])

y = torch.ones((2, 3, 4))
z = torch.randn((2, 3, 4))
print("ones     :", y.shape, "| randn:", z.shape)


In [ ]:
# C2 参考实现：按列标准化（不用 for 循环）
import torch

torch.manual_seed(0)
x = torch.randn(100, 3)

mean = x.mean(dim=0)                     # (3,)  沿样本维统计
std = x.std(dim=0, unbiased=False)       # (3,)  unbiased=False 用 n 作除数
xn = (x - mean) / std                    # (100, 3) 广播

print("shapes:", mean.shape, std.shape, xn.shape)
print("标准化后每列均值  :", xn.mean(dim=0))
print("标准化后每列标准差:", xn.std(dim=0, unbiased=False))


In [ ]:
# C3 参考实现：含 while + if 的 f(a)，梯度必须等于 d / a
import torch

def f(a):
    b = a * 2
    while b.norm() < 1000:      # 循环次数由运行时决定，动态图会逐次记录
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()
print("a =", a.item(), "| d =", d.item())
print("a.grad =", a.grad.item(), "| d / a =", (d / a).item())
assert a.grad == d / a


In [ ]:
# C4 参考实现：用 gradcheck 验证解析梯度
import torch

def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    return b if b.sum() > 0 else 100 * b

a = torch.randn(size=(), dtype=torch.double, requires_grad=True)
print("gradcheck:", torch.autograd.gradcheck(f, (a,)))
# 注意：随机点落到 while/if 的分支边界附近时可能偶发失败，换一个随机点再试


In [ ]:
# C5 参考实现：清零 -> 前向 -> 反向 -> 更新
import torch

torch.manual_seed(0)
w = torch.tensor([3.0], requires_grad=True)   # 参数，故意从错的初值开始
b = torch.tensor([2.0], requires_grad=True)
x = torch.randn(200, 1)
y = 3 * x + 2 + 0.1 * torch.randn(200, 1)     # 真值 w=3, b=2 + 噪声

for step in range(3):
    y_hat = x @ w + b                          # 前向
    loss = ((y_hat - y) ** 2).mean()
    w.grad = None                              # 清零（等价 optimizer.zero_grad()）
    b.grad = None
    loss.backward()                            # 反向
    with torch.no_grad():                      # 更新
        w -= 0.1 * w.grad
        b -= 0.1 * b.grad
    print(f"step {step}  loss={loss.item():.4f}  w={w.item():.4f}  b={b.item():.4f}")


## 第一轮 D 组｜找 bug（13 / 20）

### D1 循环里没清零 — <span style="color:#e65100">【半对】</span>

**你写的**：每次的梯度没有清 0；`x.grad().zero_()`

**诊断正确**：三轮 backward 全累加，最终 `x.grad = 3 × 4x = [0, 12, 24, 36]`。
**修法语法错**：`.grad` 是张量属性不是方法，应写 `x.grad.zero_()`，或每次迭代前 `x.grad = None`；训练循环里就是 `optimizer.zero_grad()`。

### D2 非标量 backward — ✓

**你写的**：y 是矢量不能计算梯度；`y.sum().backward()`

**修法正确**，措辞改成"backward 需要标量起点"就完美了。也可以写 `y.backward(torch.ones_like(x))`。

### D3 detach 的结果 — <span style="color:#c62828">【错】</span>

**你写的**：`y = x * x`；`y.detach()`

**判断反了**：这里不是"忘了做什么"，`.detach()` 是**你主动做的手术**——detach 之后结果本来就该是 x²，`x.grad = [0, 1, 4, 9]`。

想让结果变成 3x²，就**不能 detach**：

```python
z = (x * x * x).sum()   # z = Σx³ → ∂z/∂x = 3x² = [0, 3, 12, 27]
z.backward()
```

另外 `y.detach()` 单独写一行毫无作用——返回值没被接住，等于没写。

### D4 原地修改 — ✓

**你写的**：`Z[:] = Y + X`；`Y += X` 全对。补充写法：`Y.copy_(Y + X)`、`Y.add_(X)`。


In [ ]:
# D3 参考实现：想拿到 3x^2，就不能 detach
import torch

x = torch.arange(4.0, requires_grad=True)

# 版本 1：正常三阶（不 detach）-> 3x^2
z1 = (x * x * x).sum()
z1.backward()
print("不 detach :", x.grad.tolist())   # [0.0, 3.0, 12.0, 27.0]

# 版本 2：detach 掉 x*x -> 只剩 x 这一支贡献，梯度 = x^2
x.grad = None
z2 = ((x * x).detach() * x).sum()
z2.backward()
print("detach    :", x.grad.tolist())   # [0.0, 1.0, 4.0, 9.0]


# 第二轮｜重考卷（76 / 100）

> **对比第一轮**：57 → **76**（概念 19→24 ｜ 计算 21→23 ｜ 编程 4→15 ｜ 找 bug 13→14）
> **丢分性质变了**：第一轮是"不知道"，这一轮主要是"知道但没做完 / 没读输出"——这是两种完全不同的状态。
> **读法**：每题是「你写的 → 正确答案」；<span style="color:#c62828">红色 = 错</span>，<span style="color:#e65100">橙色 = 半对</span>。

---

## 第二轮 A 组（24 / 30）

### A1 矩阵的列与 2A — <span style="color:#e65100">【半对】</span>

**题目**：2×2 矩阵的两列是 (1,2)、(3,4)，说出这个矩阵做了什么；`2A` 与 `A` 差在哪？

**你写的**：经过变换后基向量分别落在 (1,2) 和 (3,4) 处；2A 的基向量都是 A 的两倍。

**答案**：前半对。补两件事：

- `2A` 是把每个基向量的**落点整体乘以 2**，整张图放大 2 倍；二维下面积变 **4 倍**，即 `det(2A) = 4·det(A)`。
- **`2A` ≠ `A²`**：`A²` 才是"把 A 这个变换做两次"。这是最容易混的一对。

### A2 变换顺序 — <span style="color:#e65100">【半对】</span>

**题目**：先旋转 90° 再横向拉伸 2 倍，和先拉伸再旋转一样吗？用它解释 `AB ≠ BA`。

**你写的**：不一样；这是两种不同的变换，顺序不能互换。

**答案**：结论对，但没算数。补上算例，取 î = (1,0)：

- 先旋转 90° → (0,1)，再横向拉伸 ×2 → **(0,1)**
- 先横向拉伸 ×2 → (2,0)，再旋转 90° → **(0,2)**

同一个向量两个结果，这就是 `AB ≠ BA` 的证据。以后凡是"顺序/交换律"的题，都用一个具体基向量算一遍。

### A3 三步曲 — ✓

**答案**：属于"定义损失"这一步，是**人**的工作；换损失函数不影响"参数由机器学"这件事。

### A4 四个判断 — <span style="color:#e65100">【半对】</span>

**你写的**：✗ ✗ ✓ ✗（判断全对）

**答案**：判断全对，但题目要求"改正"，要写成句子：

1. `*` 是**逐元素相乘**，不是矩阵乘法
2. `@` 是**矩阵乘法**，不是逐元素相乘
3. ✓ `torch.dot` 要求两个输入都是一维向量，结果是标量
4. `torch.cross` **不能**用于二维向量，PyTorch 要求最后一维长度为 3

### A5 cat 的合法性 — ✓

**答案**：`dim=0` 合法 → **(7,4)**；`dim=1` 报错，因为除拼接维之外其余维度必须相等，这里 3 ≠ 4。

### A6 为什么必须标量 — ✓

**答案**：链式法则需要一个起点，标量输出时起点天然是 ∂L/∂L = 1。

### A7 detach 的场景 — <span style="color:#e65100">【半对】</span>

**你写的**：①②③ 都对，**④ 漏了没答**。

**答案**：

1. 转 numpy 画图 → **必须** detach（计算图上的张量不能直接转 numpy）
2. 让中间结果不参与求导 → **必须** detach
3. 让参数永远不被训练 → 用 `requires_grad=False`，**不是** detach
4. 计算 target network 的目标值 → **必须** detach，否则梯度顺着 target 反传回去，target 会被污染

### A8 梯度累加 — ✓

**答案**：多个 loss / 分支的梯度可以直接相加（多任务训练、RNN 跨时间步、梯度累积技巧）。

---


## 第二轮 B 组（23 / 30）

### B1 三维索引 — <span style="color:#c62828">【未作答】</span>

**题目**：`X = torch.arange(24).reshape(2,3,4)`，求 `X[0]`、`X[0,1]`、`X[:,1,:]`、`X[...,0]` 的形状。

**答案**：

| 表达式 | 形状 | 为什么 |
| --- | --- | --- |
| `X[0]` | **(3,4)** | 整数索引消掉第 0 维 |
| `X[0,1]` | **(4,)** | 又消掉一维 |
| `X[:,1,:]` | **(2,4)** | 切片保留、整数索引消掉中间维 |
| `X[...,0]` | **(2,3)** | 最后一维被整数索引消掉 |

口诀：整数索引**消维**，切片（含 `:` 与 `...`）**保维**。

### B2 一维索引 — <span style="color:#c62828">【错】</span>

**题目**：`x = torch.tensor([1.,2.,3.])`，`x.shape`、`x[1].shape`、`x[1:2].shape`？

**你写的**：(3,)；(1,)；(1,1)

**答案**：

| 表达式 | 形状 |
| --- | --- |
| `x.shape` | **(3,)** ✓ |
| `x[1].shape` | **()** ← 0 维标量张量，不是 (1,) |
| `x[1:2].shape` | **(1,)** ← 切片保留维度 |

你的 (1,1) 是把"切片保留"和"再多加一维"混在了一起。要取第 1 个元素但保留维度，用 `x[1:2]` 或 `x[1].unsqueeze(0)`。

### B3 梯度累加 — ✓

**答案**：`[2,4,6]` → `[4,8,12]` → `[6,12,18]`；通式 **2k·x**。全对，第一轮的红点拿回来了。

### B4 detach 后的梯度 — ✓

**答案**：∂z/∂x = `[1,4,9]`；去掉 detach 后 = `[3,12,27]`。全对。

### B5 一维权重 — <span style="color:#e65100">【半对】</span>

**你写的**：(8,1)；(8,)；(8,8)

**答案**：三个形状都对，但要写清结论：第一种（(8,1)）与标签**正常相减**；第二种（(8,)）与 (8,1) 相减会**广播成 (8,8)**，不报错但 Loss 完全错。

### B6 广播 — ✓

**答案**：(4,5)；`sum() = 20`。

### B7 三种运算 — ✓

**答案**：`x**2 = [4,16]`；`2**x = [4,16]`；`exp(x) = [e², e⁴]`。

### B8 方向导数 — ✓

**答案**：`y.sum().backward()` → `[2,4,6]`；`y.backward(torch.tensor([1.,0.,0.]))` → `[2,0,0]`，即只取第一个分量那一路的梯度。


## 第二轮 C 组（15 / 20）

### C1 reshape — ✓

**答案**：`(12,)`、`(3,4)`、`(12,)`；reshape **不改变** x 本身，要接住返回值。

### C2 按列 min-max — <span style="color:#e65100">【半对】</span>

**你写的**：`xn = (x - min_vals) / (min_vals - max_vals)`

**答案**：分母符号写反了，应该是 **`(x - min_vals) / (max_vals - min_vals)`**。
你的输出 `每列最小值: -1`、`每列最大值: -0` 就是证据——**打印了验证，但要读它**。归一化后每列必须落在 [0,1]，这是自带断言的一步。

### C3 含 for + if/else 的函数 — <span style="color:#e65100">【半对】</span>

**你写的**：函数跑通、断言通过；手推写成 `∂d/∂a = -1`（当 a ≤ 0）。

**答案**：a ≤ 0 时循环执行的是 `a = a - 1` 十次，所以 `d = a - 10`，**∂d/∂a = 1**（不是 −1）。
因此断言里的 `== -1` 是错的——**断言必须建立在正确的手推之上**，否则它会给你虚假的安全感。

### C4 训练循环 — <span style="color:#e65100">【半对】</span>

**你写的**：循环写对了，loss 从 13.90 降到 10.14；但对照实验只在注释里写了"梯度会累积"，没真跑。

**答案**：清零用 `w.grad = None`（或 `zero_()`）都对。缺的是对照实验的结论：
不清零时 `.grad` 是**历史梯度之和**，`step()` 用的正是这个和 → 等效学习率一步步放大 → 步子越来越大，最终震荡或发散。

---


## 第二轮 D 组（14 / 20）

### D1 循环没清零 — <span style="color:#e65100">【半对】</span>

**你写的**：诊断"w 的梯度没有清零" ✓，修法 `w.grad = None` ✓，修完结果 `[0.04, 1.04]` ✓。

**问题**：你**先把 bug 修了再运行**，所以没看到错误版本长什么样——找 bug 题一半的价值在"看见错误"。

| 版本 | 输出 |
| --- | --- |
| 有 bug（不清零） | **[-1.16, -0.16]** ← 完全跑偏 |
| 修好（清零） | [0.04, 1.04] |

### D2 detach 用错 — ✓

**你写的**：诊断"不用 detach 就行" ✓；修法 `(x*3*x).sum()` ✓；正确结果 `[6,12]` ✓；还贴了错误输出 `[3,6]` ✓。第一轮的 D3 红点拿回来了。

### D3 一维权重 — ✓

**你写的**：诊断 `w` 形状应为 (3,1) ✓；两种修法（改 `w` 或 `y_hat.unsqueeze(1)`）✓；并给出了 `y_hat.shape = (16,)` 的证据 ✓。

---


## 剩余的洞（洞 1 已于 09-14 通过）

### 洞 1：索引降维 —— ✅ 09-14 已通过（用输出验证）

题目（已做过，输出见下）：

```python
x = torch.arange(24).reshape(2, 3, 4)
print(x[0].shape, x[0, 1].shape, x[:, 1, :].shape, x[..., 0].shape)
y = torch.tensor([1., 2., 3.])
print(y.shape, y[1].shape, y[1:2].shape)
```

口诀：**整数索引消维，切片保维**。`y[1]` 是 0 维 `()`，`y[1:2]` 是 1 维 `(1,)`。

**09-14 实测输出（7/7 全对）**

```
x[0]     -> torch.Size([3, 4])    整数索引消掉第 0 维
x[0,1]   -> torch.Size([4])       再消一维
x[:,1,:] -> torch.Size([2, 4])    切片保维、整数索引消维
x[...,0] -> torch.Size([2, 3])    最后一维被消掉
y        -> torch.Size([3])
y[1]     -> torch.Size([])        0 维标量张量
y[1:2]   -> torch.Size([1])       切片保维
```

**两个补充（面试会问）**

- `y[1]` 虽然是 0 维，但**仍然带 `grad_fn`**——这正是 `loss.backward()` 能直接在 0 维 loss 上工作的原因；想脱离计算图要用 `.item()` 或 `float()`。
- 喂单个样本时 `model(x[0])` 会把 batch 维消掉直接报错，正确写法是 `model(x[0:1])`。


### 洞 2：验证意识（打印了就要读）

C2 的输出里 `min = -1` 已经把手举起来了，但没被看见。
新规矩：凡是打印了 min / max / shape / loss 的地方，**下一行必须写一句"这说明什么"**，不写不算完成。

### 洞 3：对照实验必须真跑

C4 的"不清零会怎样"、D1 的"错误版本输出是什么"都属于这类。
新规矩：题目里出现"如果…会怎样"，**必须把那行代码改掉重跑**，并贴出输出。

### 洞 4：结论要数值化

A1 的 `det(2A) = 4det(A)`、A2 的 `(0,1)` vs `(0,2)`、A7 的 ④ target network——都是"知道大概"但没落到具体数字或具体场景。补上这三处就齐了。

---


## 重做清单（只剩这些）

<span style="color:#22a06b">**已验证通过**</span>：B1、B2（索引降维）—— 09-14 实测输出 7/7 正确

<span style="color:#c62828">**必须重做**</span>：C2（符号 + 读输出）｜D1（跑错误版本）

<span style="color:#e65100">**补齐即可**</span>：A1（det(2A)）｜A2（具体算例）｜A4（写成句子）｜A7 ④（target network）｜B5（写结论）｜C3（负支梯度 = 1）｜C4（跑对照）

**已经拿回的红点**：A4 逐元素 vs 矩阵乘 ｜ B3 梯度累加 ｜ C1 / C4 训练循环 ｜ D2 / D3 找 bug

**一句话**：一轮涨了 19 分，剩下的问题只有两类——"降维没形成条件反射"和"实验纪律不够"。这两个都不是能力问题，是练习次数问题。
